# Random Forest Benchmark на Sales Dataset

Бенчмаркинг Random Forest классификатора с мониторингом производительности CPU на датасете продаж.

**Задача:** Классификация типа клиента (New/Returning) на основе данных о продажах

**Что измеряется:**
- ⏱️ Время обучения Random Forest с разным количеством деревьев (50, 100, 200)
- 💻 Загрузка CPU во время обучения
- 📊 Распределение нагрузки по ядрам CPU
- 🎯 Точность бинарной классификации

**Технические детали:**
- Датасет: Sales Dataset (1,000 записей о продажах)
- Признаки: 14 (категориальные + числовые)
- Модель: RandomForestClassifier с `n_jobs=-1` (многопоточность)
- Метрики: время обучения, точность, загрузка CPU
- Количество запусков на конфигурацию: 3 (для усреднения)

**Примечание:** Random Forest в sklearn использует многопоточность (threads), поэтому все вычисления происходят в одном процессе Python с параллельными потоками.

In [1]:
import numpy as np
import pandas as pd
import time
import threading
import psutil
import os
import sys
import subprocess
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# Настройка отображения
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

In [13]:
class ProcessCPUMonitor:
    """Мониторинг загрузки CPU только для процессов бенчмарка"""
    
    def __init__(self, interval=1.0):
        self.interval = interval
        self.monitoring = False
        self.process_data = []
        self.benchmark_pids = set()
        self.thread = None
        self.main_pid = os.getpid()
    
    def start_monitoring(self):
        """Запуск мониторинга в отдельном потоке"""
        self.monitoring = True
        self.process_data = []
        self.benchmark_pids = set()
        self.benchmark_pids.add(self.main_pid)
        self.thread = threading.Thread(target=self._monitor_loop)
        self.thread.daemon = True
        self.thread.start()
    
    def _get_benchmark_processes(self):
        """Получение всех процессов, связанных с бенчмарком"""
        current_benchmark_pids = set()
        
        try:
            # Поиск дочерних процессов
            parent = psutil.Process(self.main_pid)
            children = parent.children(recursive=True)
            for child in children:
                current_benchmark_pids.add(child.pid)
            
            # Добавляем основной процесс
            current_benchmark_pids.add(self.main_pid)
            
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
        
        return current_benchmark_pids
    
    def _monitor_loop(self):
        """Цикл мониторинга"""
        # Первый вызов cpu_percent для инициализации
        for pid in list(self.benchmark_pids):
            try:
                process = psutil.Process(pid)
                process.cpu_percent()  # Инициализирующий вызов
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        
        time.sleep(0.1)  # Небольшая пауза для инициализации
        
        while self.monitoring:
            try:
                timestamp = time.time()
                current_pids = self._get_benchmark_processes()
                self.benchmark_pids.update(current_pids)
                
                total_cpu_percent = 0
                per_process_data = {}
                per_core_usage = {}
                
                # Мониторинг каждого процесса бенчмарка
                for pid in list(self.benchmark_pids):
                    try:
                        process = psutil.Process(pid)
                        
                        # ВАЖНО: cpu_percent(interval=None) использует предыдущее значение
                        # Поэтому вызываем с небольшим интервалом для точности
                        cpu_percent = process.cpu_percent(interval=0.1)
                        
                        with process.oneshot():
                            memory_info = process.memory_info()
                            
                            # Получение использования по ядрам (если доступно)
                            cpu_times = process.cpu_times()
                            cpu_affinity = process.cpu_affinity() if hasattr(process, 'cpu_affinity') else []
                            
                            total_cpu_percent += cpu_percent
                            
                            per_process_data[pid] = {
                                'name': process.name(),
                                'cpu_percent': cpu_percent,
                                'memory_rss': memory_info.rss,
                                'cpu_times': cpu_times._asdict(),
                                'cpu_affinity': cpu_affinity
                            }
                            
                    except (psutil.NoSuchProcess, psutil.AccessDenied):
                        # Удаляем завершенные процессы
                        self.benchmark_pids.discard(pid)
                        continue
                
                # Мониторинг общего использования CPU по ядрам для процессов бенчмарка
                system_cpu_percent = psutil.cpu_percent(interval=None, percpu=True)
                
                self.process_data.append({
                    'timestamp': timestamp,
                    'total_cpu_percent': total_cpu_percent,
                    'per_process_data': per_process_data,
                    'system_cpu_percent': system_cpu_percent,
                    'active_pids': list(self.benchmark_pids)
                })
                
                time.sleep(self.interval)
                
            except Exception as e:
                print(f"Ошибка мониторинга: {e}")
                time.sleep(self.interval)
    
    def stop_monitoring(self):
        """Остановка мониторинга"""
        self.monitoring = False
        if self.thread:
            self.thread.join(timeout=2)
        return self.get_summary()
    
    def get_summary(self):
        """Получение сводки по использованию CPU"""
        if not self.process_data:
            return {}
        
        total_cpu_usage = [data['total_cpu_percent'] for data in self.process_data]
        
        # Анализ по процессам
        process_analysis = {}
        for data in self.process_data:
            for pid, proc_data in data['per_process_data'].items():
                if pid not in process_analysis:
                    process_analysis[pid] = {
                        'name': proc_data['name'],
                        'cpu_percent': [],
                        'memory_rss': []
                    }
                process_analysis[pid]['cpu_percent'].append(proc_data['cpu_percent'])
                process_analysis[pid]['memory_rss'].append(proc_data['memory_rss'])
        
        # Статистика по процессам
        process_stats = {}
        for pid, data in process_analysis.items():
            process_stats[pid] = {
                'name': data['name'],
                'avg_cpu_percent': np.mean(data['cpu_percent']),
                'max_cpu_percent': np.max(data['cpu_percent']),
                'avg_memory_mb': np.mean(data['memory_rss']) / 1024 / 1024,
                'max_memory_mb': np.max(data['memory_rss']) / 1024 / 1024
            }
        
        # Анализ использования по ядрам
        core_analysis = {}
        if self.process_data and 'system_cpu_percent' in self.process_data[0]:
            num_cores = len(self.process_data[0]['system_cpu_percent'])
            for core in range(num_cores):
                core_usage = [data['system_cpu_percent'][core] for data in self.process_data]
                core_analysis[core] = {
                    'avg_usage': np.mean(core_usage),
                    'max_usage': np.max(core_usage),
                    'min_usage': np.min(core_usage)
                }
        
        return {
            'duration_seconds': self.process_data[-1]['timestamp'] - self.process_data[0]['timestamp'],
            'total_avg_cpu_usage': np.mean(total_cpu_usage),
            'total_max_cpu_usage': np.max(total_cpu_usage),
            'process_stats': process_stats,
            'core_analysis': core_analysis,
            'sample_count': len(self.process_data),
            'max_concurrent_processes': max([len(data['active_pids']) for data in self.process_data])
        }

# Информация о системе
def get_system_info():
    """Получение информации о системе"""
    cpu_info = {
        'physical_cores': psutil.cpu_count(logical=False),
        'logical_cores': psutil.cpu_count(logical=True),
        'cpu_freq': psutil.cpu_freq()._asdict() if psutil.cpu_freq() else {},
        'architecture': os.uname().machine
    }
    
    memory_info = psutil.virtual_memory()._asdict()
    
    return {
        'cpu': cpu_info,
        'memory': memory_info,
        'timestamp': datetime.now().isoformat(),
        'python_version': f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
    }

In [ ]:
# Информация о системе
system_info = get_system_info()
print("СИСТЕМНАЯ ИНФОРМАЦИЯ")
print("="*60)
print(f"Логических ядер: {system_info['cpu']['logical_cores']}")
print(f"Физических ядер: {system_info['cpu']['physical_cores']}")
print(f"Архитектура: {system_info['cpu']['architecture']}")
print(f"Память: {system_info['memory']['total'] / (1024**3):.1f} ГБ")
print(f"Python: {system_info['python_version']}")
print("="*60)


Система: 8 логических ядер
Архитектура: x86_64
Память: 14.9 ГБ

1. ЗАГРУЗКА ДАННЫХ MNIST...
Данные загружены: 60000 тренировочных, 10000 тестовых образцов
Данные загружены: 60000 тренировочных, 10000 тестовых образцов


In [ ]:
# Конфигурация бенчмарка
class Config:
    RF_ESTIMATORS = [50, 100, 200]
    MAX_DEPTH = 20
    NUM_RUNS = 3
    RANDOM_STATE = 42

config = Config()

# Функция тестирования Random Forest с мониторингом CPU
def benchmark_random_forest(X_train, y_train, X_test, y_test, n_estimators=100, max_depth=20):
    """Запуск бенчмарка для Random Forest с мониторингом CPU"""
    
    times = []
    accuracies = []
    process_monitor_data = []
    
    for run in range(config.NUM_RUNS):
        print(f"  Запуск {run + 1}/{config.NUM_RUNS}...", end=' ')
        
        # Запуск мониторинга CPU
        process_monitor = ProcessCPUMonitor(interval=0.5)
        process_monitor.start_monitoring()
        
        start_time = time.perf_counter()
        
        # Создание и обучение модели
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=config.RANDOM_STATE + run,
            n_jobs=-1  # Использование всех ядер
        )
        
        rf.fit(X_train, y_train)
        training_time = time.perf_counter() - start_time
        
        # Остановка мониторинга и получение данных
        process_summary = process_monitor.stop_monitoring()
        
        # Оценка точности
        y_pred = rf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        times.append(training_time)
        accuracies.append(accuracy)
        process_monitor_data.append(process_summary)
        
        print(f"Время: {training_time:.3f}с, Точность: {accuracy:.4f}, CPU: {process_summary.get('total_avg_cpu_usage', 0):.1f}%")
    
    return {
        'times': times,
        'accuracies': accuracies,
        'process_monitor_data': process_monitor_data,
        'avg_time': np.mean(times),
        'std_time': np.std(times),
        'avg_accuracy': np.mean(accuracies),
        'std_accuracy': np.std(accuracies),
        'avg_cpu_usage': np.mean([data.get('total_avg_cpu_usage', 0) for data in process_monitor_data])
    }


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vinothkannaece/sales-dataset")

print("Path to dataset files:", path)


/home/batoshka/ABC_kursach/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 27.0k/27.0k [00:00<00:00, 142kB/s]

Extracting files...


Path to dataset files: /home/batoshka/.cache/kagglehub/datasets/vinothkannaece/sales-dataset/versions/1


In [26]:
# Загрузка и анализ датасета продаж
data = pd.read_csv(f"{path}/sales_data.csv")
print(f"Размер датасета: {data.shape}")
print(f"\nПервые строки:")
print(data.head())
print(f"\nИнформация о столбцах:")
print(data.info())
print(f"\nСтатистика:")
print(data.describe())
print(f"\nПропущенные значения:")
print(data.isnull().sum())

Размер датасета: (1000, 14)

Первые строки:
   Product_ID   Sale_Date Sales_Rep Region  Sales_Amount  Quantity_Sold  \
0        1052  2023-02-03       Bob  North       5053.97             18   
1        1093  2023-04-21       Bob   West       4384.02             17   
2        1015  2023-09-21     David  South       4631.23             30   
3        1072  2023-08-24       Bob  South       2167.94             39   
4        1061  2023-03-24   Charlie   East       3750.20             13   

  Product_Category  Unit_Cost  Unit_Price Customer_Type  Discount  \
0        Furniture     152.75      267.22     Returning      0.09   
1        Furniture    3816.39     4209.44     Returning      0.11   
2             Food     261.56      371.40     Returning      0.20   
3         Clothing    4330.03     4467.75           New      0.02   
4      Electronics     637.37      692.71           New      0.08   

  Payment_Method Sales_Channel Region_and_Sales_Rep  
0           Cash        Online      

In [ ]:
# Запуск бенчмарка
print("\nБЕНЧМАРК RANDOM FOREST")
print("="*60)
results = {}

for n_est in config.RF_ESTIMATORS:
    print(f"\nТестирование с {n_est} деревьями:")
    
    result = benchmark_random_forest(X_train, y_train, X_test, y_test, n_est, config.MAX_DEPTH)
    results[n_est] = result
    
    print(f"  Среднее время: {result['avg_time']:.3f} ± {result['std_time']:.3f} сек")
    print(f"  Средняя точность: {result['avg_accuracy']:.4f} ± {result['std_accuracy']:.4f}")
    print(f"  Средняя загрузка CPU: {result['avg_cpu_usage']:.1f}%")

print("\n✓ Тестирование завершено!")


In [ ]:
# Визуализация результатов
print("\nВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*60)

estimators = list(results.keys())
times = [results[est]['avg_time'] for est in estimators]
time_std = [results[est]['std_time'] for est in estimators]
cpu_usage = [results[est]['avg_cpu_usage'] for est in estimators]
accuracies = [results[est]['avg_accuracy'] * 100 for est in estimators]

print(f"Деревья: {estimators}")
print(f"Времена (сек): {[f'{t:.3f}' for t in times]}")
print(f"Точность (%): {[f'{a:.2f}' for a in accuracies]}")
print(f"CPU (%): {[f'{c:.1f}' for c in cpu_usage]}")

# Создание графиков
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Производительность Random Forest на Sales Dataset', fontsize=16, fontweight='bold')

# График 1: Время обучения
bars1 = ax1.bar(range(len(estimators)), times, yerr=time_std, 
                capsize=5, alpha=0.7, color='coral', edgecolor='darkred')
ax1.set_xlabel('Количество деревьев', fontsize=12)
ax1.set_ylabel('Время выполнения (сек)', fontsize=12)
ax1.set_title('Время обучения', fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(estimators)))
ax1.set_xticklabels(estimators)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(bottom=0, top=max(times) * 1.15 if times else 1)

for i, (bar, value) in enumerate(zip(bars1, times)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max(times) if times else 0)*0.02, 
             f'{value:.3f}±{time_std[i]:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# График 2: Загрузка CPU
bars2 = ax2.bar(range(len(estimators)), cpu_usage, alpha=0.7, 
                color='mediumpurple', edgecolor='indigo')
ax2.set_xlabel('Количество деревьев', fontsize=12)
ax2.set_ylabel('Загрузка CPU (%)', fontsize=12)
ax2.set_title('Средняя загрузка CPU', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(estimators)))
ax2.set_xticklabels(estimators)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(bottom=0, top=min(800, max(cpu_usage) * 1.2) if cpu_usage and max(cpu_usage) > 0 else 100)

for i, (bar, value) in enumerate(zip(bars2, cpu_usage)):
    ax2.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + (max(cpu_usage) if cpu_usage and max(cpu_usage) > 0 else 0)*0.02, 
             f'{value:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# График 3: Точность классификации
bars3 = ax3.bar(range(len(estimators)), accuracies, alpha=0.7, 
                color='lightseagreen', edgecolor='teal')
ax3.set_xlabel('Количество деревьев', fontsize=12)
ax3.set_ylabel('Точность (%)', fontsize=12)
ax3.set_title('Точность классификации', fontsize=12, fontweight='bold')
ax3.set_xticks(range(len(estimators)))
ax3.set_xticklabels(estimators)
ax3.grid(True, alpha=0.3, axis='y')
if accuracies:
    ax3.set_ylim(bottom=min(accuracies) * 0.95, top=max(accuracies) * 1.02)

for i, (bar, value) in enumerate(zip(bars3, accuracies)):
    y_offset = (max(accuracies) - min(accuracies))*0.01 if accuracies else 1
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + y_offset, 
             f'{value:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Графики построены!")


In [ ]:
# Анализ распределения нагрузки по ядрам CPU
print("\nАНАЛИЗ РАСПРЕДЕЛЕНИЯ НАГРУЗКИ ПО ЯДРАМ CPU")
print("="*60)

# Используем результаты с максимальной загрузкой CPU
best_estimator = estimators[np.argmax([results[est]['avg_cpu_usage'] for est in estimators])]
best_result = results[best_estimator]

print(f"Анализируем конфигурацию с максимальной загрузкой: {best_estimator} деревьев")
print(f"Средняя загрузка CPU: {best_result['avg_cpu_usage']:.1f}%\n")

fig, ax = plt.subplots(figsize=(14, 6))

# Сбор данных по использованию ядер
core_usage_data = []
for run_data in best_result['process_monitor_data']:
    core_analysis = run_data.get('core_analysis', {})
    for core, stats in core_analysis.items():
        core_usage_data.append({
            'core': core,
            'usage': stats['avg_usage']
        })

if core_usage_data:
    core_df = pd.DataFrame(core_usage_data)
    core_usage_avg = core_df.groupby('core')['usage'].mean()
    
    cores = list(core_usage_avg.index)
    usage_values = core_usage_avg.values
    
    # Группировка ядер по физическим core (если есть hyper-threading)
    physical_cores = system_info['cpu']['physical_cores']
    logical_cores = system_info['cpu']['logical_cores']
    
    print(f"Система: {physical_cores} физических ядер, {logical_cores} логических ядер")
    
    if logical_cores > physical_cores:  # Hyper-threading включен
        cores_per_physical = logical_cores // physical_cores
        heatmap_data = []
        for i in range(physical_cores):
            physical_core_usage = []
            for j in range(cores_per_physical):
                core_idx = i * cores_per_physical + j
                if core_idx < len(usage_values):
                    physical_core_usage.append(usage_values[core_idx])
            heatmap_data.append(physical_core_usage)
        
        im = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
        ax.set_xlabel('Логические потоки (threads)', fontsize=12)
        ax.set_ylabel('Физические ядра', fontsize=12)
        ax.set_title(f'Распределение нагрузки по ядрам CPU ({best_estimator} деревьев)\nHyper-Threading: {cores_per_physical} потока на ядро', 
                     fontsize=13, fontweight='bold')
        ax.set_xticks(range(cores_per_physical))
        ax.set_xticklabels([f'Thread {i}' for i in range(cores_per_physical)])
        ax.set_yticks(range(physical_cores))
        ax.set_yticklabels([f'Core {i}' for i in range(physical_cores)])
        
        # Добавление значений
        for i in range(len(heatmap_data)):
            for j in range(len(heatmap_data[i])):
                text_color = "white" if heatmap_data[i][j] > 50 else "black"
                ax.text(j, i, f'{heatmap_data[i][j]:.1f}%', 
                       ha="center", va="center", color=text_color, fontsize=10, fontweight='bold')
        
        plt.colorbar(im, ax=ax, label='Загрузка CPU (%)')
        
        # Статистика
        avg_usage = np.mean([v for row in heatmap_data for v in row])
        max_usage = np.max([v for row in heatmap_data for v in row])
        min_usage = np.min([v for row in heatmap_data for v in row])
        
        print(f"\nСтатистика по ядрам:")
        print(f"  Средняя загрузка: {avg_usage:.1f}%")
        print(f"  Максимальная загрузка: {max_usage:.1f}%")
        print(f"  Минимальная загрузка: {min_usage:.1f}%")
        print(f"  Разброс: {max_usage - min_usage:.1f}%")
    
    else:
        # Нет hyper-threading - простой bar chart
        bars = ax.bar(cores, usage_values, alpha=0.7, color='coral', edgecolor='darkred', width=0.7)
        ax.set_xlabel('Номер ядра CPU', fontsize=12)
        ax.set_ylabel('Загрузка (%)', fontsize=12)
        ax.set_title(f'Загрузка ядер CPU ({best_estimator} деревьев)', fontsize=13, fontweight='bold')
        ax.set_xticks(cores)
        ax.set_xticklabels([f'{i}' for i in cores])
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim(0, 100)
        
        # Добавление значений
        for bar, value in zip(bars, usage_values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                   f'{value:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        print(f"\nСтатистика по ядрам:")
        print(f"  Средняя загрузка: {np.mean(usage_values):.1f}%")
        print(f"  Максимальная загрузка: {np.max(usage_values):.1f}%")
        print(f"  Минимальная загрузка: {np.min(usage_values):.1f}%")

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Нет данных об использовании ядер CPU")


In [ ]:
# Итоговый отчет
print("\n" + "="*80)
print("ИТОГОВЫЙ ОТЧЕТ: RANDOM FOREST НА SALES DATASET")
print("="*80)

print(f"\nСИСТЕМНАЯ ИНФОРМАЦИЯ:")
print(f"  Логических ядер: {system_info['cpu']['logical_cores']}")
print(f"  Физических ядер: {system_info['cpu']['physical_cores']}")
print(f"  Архитектура: {system_info['cpu']['architecture']}")
print(f"  Память: {system_info['memory']['total'] / (1024**3):.1f} ГБ")

print(f"\nДАТАСЕТ:")
print(f"  Название: Sales Dataset")
print(f"  Размер: {len(data)} записей")
print(f"  Признаков: {len(feature_cols)}")
print(f"  Train/Test: {len(X_train)}/{len(X_test)}")
print(f"  Задача: Бинарная классификация (New/Returning Customer)")

print(f"\nРЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ:")
print(f"{'Деревьев':<10} {'Время (сек)':<15} {'Точность (%)':<15} {'CPU (%)':<12} {'Эффективность':<15}")
print("-" * 80)

for n_est in estimators:
    time_val = results[n_est]['avg_time']
    accuracy_val = results[n_est]['avg_accuracy'] * 100
    cpu_val = results[n_est]['avg_cpu_usage']
    efficiency = accuracy_val / time_val if time_val > 0 else 0
    
    print(f"{n_est:<10} {time_val:<15.3f} {accuracy_val:<15.2f} {cpu_val:<12.1f} {efficiency:<15.1f}")

avg_cpu = np.mean([results[est]['avg_cpu_usage'] for est in estimators])
max_cpu = max([results[est]['avg_cpu_usage'] for est in estimators])

print(f"\nАНАЛИЗ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"  Средняя загрузка CPU: {avg_cpu:.1f}%")
print(f"  Максимальная загрузка CPU: {max_cpu:.1f}%")
print(f"  Эффективность использования ядер: {max_cpu / system_info['cpu']['logical_cores']:.1f}% на ядро")

best_acc_idx = np.argmax([results[est]['avg_accuracy'] for est in estimators])
fastest_idx = np.argmin([results[est]['avg_time'] for est in estimators])
most_eff_idx = np.argmax([results[est]['avg_accuracy'] / results[est]['avg_time'] for est in estimators])

print(f"\nОПТИМАЛЬНЫЕ ПАРАМЕТРЫ:")
print(f"  🎯 Лучшая точность: {results[estimators[best_acc_idx]]['avg_accuracy']*100:.2f}% ({estimators[best_acc_idx]} деревьев)")
print(f"  ⚡ Самое быстрое обучение: {results[estimators[fastest_idx]]['avg_time']:.3f} сек ({estimators[fastest_idx]} деревьев)")
print(f"  ⭐ Наиболее эффективное: {estimators[most_eff_idx]} деревьев")

print(f"\nВЫВОДЫ:")
print("1. Random Forest эффективно обучается на небольших табличных данных")
print("2. Время обучения растет линейно с количеством деревьев")
print("3. Точность классификации стабильна для всех конфигураций")
print("4. Многопоточность эффективно распределяет нагрузку по CPU ядрам")
print(f"5. Оптимальный баланс точности/скорости: {estimators[most_eff_idx]} деревьев")

print("\n" + "="*80)
print("Тестирование завершено!")
print("="*80)

# Сохранение отчета
report = {
    'system_info': system_info,
    'dataset': {
        'name': 'Sales Dataset',
        'size': len(data),
        'features': len(feature_cols),
        'train_size': len(X_train),
        'test_size': len(X_test)
    },
    'config': {
        'estimators': config.RF_ESTIMATORS,
        'max_depth': config.MAX_DEPTH,
        'num_runs': config.NUM_RUNS,
        'random_state': config.RANDOM_STATE
    },
    'results': results,
    'summary': {
        'avg_cpu': avg_cpu,
        'max_cpu': max_cpu,
        'best_accuracy': results[estimators[best_acc_idx]]['avg_accuracy'],
        'best_accuracy_trees': estimators[best_acc_idx],
        'fastest_time': results[estimators[fastest_idx]]['avg_time'],
        'fastest_trees': estimators[fastest_idx],
        'most_efficient_trees': estimators[most_eff_idx]
    }
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_filename = f'sales_rf_benchmark_{timestamp}.json'

with open(report_filename, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"\n📄 Детальный отчет сохранен: {report_filename}")


In [ ]:
RandomForestClassifier()